# W4 실습 — ReAct 루프 구현

**학습 목표**

- 형식 계약(시스템 프롬프트)이 모델 출력을 어떻게 구조화하는지 실행으로 확인한다
- 모델이 Observation을 지어내는 문제를 재현하고, 루프가 이를 어떻게 차단하는지 이해한다
- `labs/docqa/loop.py`의 `parse_step()`을 구현하고 ReAct 루프를 수동·자동으로 실행한다
- 형식 준수율을 측정하고, 시스템 프롬프트를 개선해 준수율을 높인다

| 절 | 내용 | 실행 환경 |
|---|---|---|
| 1 | 준비 — 경로·모델 연결 | 모델 필요 |
| 2 | 형식 계약 — 계약 유무 비교 | 모델 필요 |
| 3 | Observation 환각 실험 | 모델 필요 |
| 4 | `parse_step` 구현·검증 | 오프라인 |
| 5 | 루프 한 바퀴 수동 실행 | 모델 필요 |
| 6 | `react_loop` 자동화·테스트 | 오프라인 + 모델 |
| 7 | 프롬프트 개선 과제 — 형식 준수율 | 모델 필요 |
| 8 | 연습문제 | 혼합 |

선행 조건: W1–W3 실습 완료 (`llm.py`의 `chat()`, `reasoning.py`, `tools.py`의 레지스트리 · `.env`의 API 키 또는 Ollama).
완료 기준: `pytest tests/test_week04.py` 전체 통과 + 7절 준수율 개선.

> 구성 참조: Hugging Face *agents-course* Unit 1 (dummy agent — Observation 환각 실험) · DeepLearning.AI *Agentic AI* (프롬프트 작성 중심의 과제 형식). 원본은 `materials/repos/`에 있다.


## 1. 준비 — 경로와 모델 연결

이 노트북은 `lectures/week04/`에 있으므로 `labs/`를 import 경로에 추가한다.
`autoreload`는 `loop.py`를 수정할 때마다 커널 재시작 없이 변경을 반영한다.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

LABS = Path.cwd().resolve().parents[1] / "labs"
assert LABS.is_dir(), f"labs 디렉터리를 찾지 못함: {LABS}"
sys.path.insert(0, str(LABS))

from docqa import llm, loop


W1에서 구현한 `chat()`의 연결을 확인한다. 오류가 나면 W1 실습(`docqa/llm.py`의 TODO)과 `.env` 설정을 먼저 완료한다.


In [ ]:
print(llm.chat([{"role": "user", "content": "1+1=? 답만."}]))


## 2. 형식 계약 — 계약이 없을 때와 있을 때

**도구(tool)** = 모델 외부에 존재하는 실행 가능한 함수·API. 모델은 도구 호출을 텍스트로 선언할 뿐, 실행은 모델 바깥의 코드가 담당한다 (개념 정의는 이론 덱 참조). 호출 메커니즘과 도구 구현은 W3에서 완성했다. 이 절과 3절의 실험은 도구를 등록하지 않은 상태로 진행한다 — 형식 계약 자체의 효과를 분리해 관찰하기 위해서다. 등록된 도구와의 결합은 6절에서 확인한다.

루프는 모델 출력이 정해진 형식을 따른다는 전제 위에서만 작동한다. 그 형식을 시스템 프롬프트로 강제하며, 이를 모델과 루프 사이의 형식 계약이라 부른다. 계약의 내용: 모델은 매 턴 `Action`(도구 호출 선언) 또는 `Final Answer`(답 확정) 중 하나만 생성하고, `Observation`(도구 실행 결과)은 모델이 아니라 제어 루프(우리 코드)가 채운다.


In [ ]:
print(loop.SYSTEM_PROMPT)


계약의 효과를 확인하기 위해 같은 질문을 계약 없이, 그리고 계약과 함께 보낸다.


In [ ]:
QUESTION = "서울에서 부산까지 KTX 소요 시간은? 필요하면 검색해서 답하라."

# (1) 계약 없음 — 자유 형식 산문
free = llm.chat([{"role": "user", "content": QUESTION}])
print("── 계약 없음 ──")
print(free)

# (2) 계약 있음 — Thought / Action 형식
bound = llm.chat([
    {"role": "system", "content": loop.SYSTEM_PROMPT},
    {"role": "user", "content": f"Question: {QUESTION}"},
])
print("\n── 계약 있음 ──")
print(bound)


계약이 없으면 출력이 자유 산문이라 프로그램이 다음 행동을 결정할 수 없다. 계약이 있으면 출력이 `Thought:`/`Action:` 구조를 따르므로 파싱이 가능해진다.


## 3. Observation 환각 실험 — 루프가 생성을 끊어야 하는 이유

계약에는 "Observation은 시스템이 채워 준다 — 직접 쓰지 마라"는 문장이 있다. 이 문장이 없으면 무엇이 일어나는지 확인한다. 이 실험은 HF agents-course Unit 1의 dummy agent 실습을 따른 것이다.


In [ ]:
GUARD_LINE = "Observation: 은 시스템이 채워 준다 — 직접 쓰지 마라. Action 뒤에는 아무것도 쓰지 마라."
assert GUARD_LINE in loop.SYSTEM_PROMPT
no_guard_prompt = loop.SYSTEM_PROMPT.replace(GUARD_LINE, "")

out = llm.chat([
    {"role": "system", "content": no_guard_prompt},
    {"role": "user", "content": "Question: 지금 런던 날씨는? 날씨 도구를 사용해 확인하라."},
])
print(out)


출력에서 `Action:` 뒤에 모델이 스스로 `Observation:`을 이어 쓰고 그 위에 Final Answer까지 세우는 사례가 관찰된다 (모델·온도에 따라 재현되지 않을 수 있다 — 여러 번 실행해 본다). 도구는 실행된 적이 없으므로 이 Observation은 도구 결과의 환각이며, 그 위의 답은 접지되지 않은 답이다.

방어는 두 겹이다.

1. **프롬프트 수준** — 계약의 해당 문장이 Action 뒤 생성을 멈추도록 지시한다
2. **파서 수준** — 다음 절에서 구현할 `parse_step()`이 Action까지만 취하고, Observation은 루프가 실제 도구 결과로 채운다

이 두 장치가 "모델은 제안하고, 실행과 사실 확인은 루프가 한다"는 역할 분리를 강제한다.


## 4. `parse_step` 구현 — 모델 출력의 판정 (오프라인)

`parse_step()`은 모델 출력 한 턴을 세 경우로 판정한다.

| 반환 | 조건 |
|---|---|
| `("final", 답)` | `Final Answer:`가 있는 경우 (최우선) |
| `("action", 도구, 입력)` | `Action:` 뒤의 JSON 객체가 파싱되는 경우 |
| `("neither", 원문)` | 둘 다 아니거나 JSON이 깨진 경우 |

`labs/docqa/loop.py`를 열어 `TODO(W4)`를 구현한다 (힌트가 docstring에 있다). 구현 전에는 아래 셀에서 `NotImplementedError`가 발생하며, 파일을 저장하면 autoreload가 즉시 반영한다.


In [ ]:
CASES = [
    ("Thought: 계산이 끝났다.\nFinal Answer: 42",
     ("final", "42")),
    ('Thought: 계산기가 필요하다.\nAction: {"tool": "calculator", "input": "400/1400"}',
     ("action", "calculator", "400/1400")),
    ("음… 잘 모르겠다.",
     None),                               # 기대: kind == "neither"
    ('Action: {"tool": calculator}',      # 따옴표 없는 JSON — 파싱 실패
     None),                               # 기대: kind == "neither"
]

for text, expected in CASES:
    got = loop.parse_step(text)
    ok = (got == expected) if expected else (got[0] == "neither")
    print(f"[{'OK' if ok else 'FAIL'}] 입력: {text.splitlines()[-1][:48]!r}")
    print(f"       판정: {got}")


네 경우 모두 OK이면 판정기가 완성된 것이다. `neither`는 버리는 경로가 아니라 재요청 경로다 — 루프가 형식 오류를 알리는 Observation을 돌려보내 모델이 형식을 교정할 기회를 얻는다 (6절에서 확인).


## 5. 루프 한 바퀴 수동 실행

`react_loop()`를 쓰기 전에, 루프가 하는 일을 손으로 한 바퀴 수행한다. 핵심 자료구조는 `messages` 리스트 하나이며, 루프의 실체는 이 리스트에 항목을 추가하며 모델을 반복 호출하는 것이 전부다.

**턴 1** — 계약과 질문으로 시작해 모델의 첫 출력을 받고 판정한다.


In [ ]:
messages = [
    {"role": "system", "content": loop.SYSTEM_PROMPT},
    {"role": "user", "content": "Question: 1400의 29%는 얼마인가?"},
]

out1 = llm.chat(messages)
print(out1)
print("\n판정:", loop.parse_step(out1))


계산 문제이므로 모델은 대개 계산기 도구를 호출하려 한다 (`action` 판정). 바로 답했다면 (`final`) 아래 셀은 그 출력 기준으로 읽는다.

**턴 2** — 루프의 역할을 수행한다: 모델 출력을 대화에 추가하고, Observation을 만들어 되먹인다. 3절에서 본 환각과 달리, 여기서는 Observation을 모델 바깥(우리)이 작성한다. 아직 도구가 없으므로 도구 부재를 알리는 내용이다.


In [ ]:
messages.append({"role": "assistant", "content": out1})
messages.append({"role": "user", "content":
    "Observation: (아직 도구가 없습니다. 아는 범위에서 추론해 Final Answer를 내세요.)"})

out2 = llm.chat(messages)
print(out2)
print("\n판정:", loop.parse_step(out2))


모델이 Final Answer로 전환하면 한 바퀴가 끝난 것이다. 손으로 수행한 절차를 정리하면:

1. 모델 출력을 받아 판정한다 (`parse_step`)
2. `final`이면 종료, `action`이면 도구를 실행해 Observation을 만든다
3. 출력과 Observation을 `messages`에 추가하고 다시 호출한다

이 세 단계를 while 문에 넣은 것이 `react_loop()`다. `messages` 전체를 출력하면 대화가 어떻게 자랐는지 보인다.


In [ ]:
for m in messages:
    print(f"--- {m['role']} ---")
    print(m["content"][:200], "\n")


## 6. `react_loop` 자동화와 테스트

`loop.py`의 `react_loop()`(제공 코드)가 5절의 수동 절차를 자동화한다. 코드를 열어 다음을 확인한다.

- `for step in range(1, max_steps + 1)` — 무한 루프 방지 상한
- 판정에 따른 분기: `final` → 반환 / `action` → `dispatch()` / `neither` → 형식 오류 Observation
- `dispatch()` — 도구 실행. W3의 `tools.py` 레지스트리에 등록된 도구를 실행하며, 미등록 도구 호출은 오류 관찰로 되먹인다

아래 셀은 W3의 기본 도구를 등록한 뒤 실제 모델로 실행한다. 루프가 계산기를 실제로 호출해 답하면 첫 완전한 에이전트가 완성된 것이다.


In [ ]:
from docqa import tools

tools.register_defaults()  # W3 기본 도구(calculator·text_search) 등록
answer = loop.react_loop("1400의 29%는 얼마인가?")
print("\n최종 답:", answer)


`react_loop()`는 `llm_fn` 인자로 모델 호출 함수를 주입받는다. 정해진 출력을 순서대로 반환하는 가짜 함수를 주입하면 API 키 없이 제어 흐름만 검증할 수 있다. `tests/test_week04.py`가 같은 기법을 사용한다.

형식을 지키지 않는 모델을 주입하면 `neither` 경로의 방어도 확인할 수 있다: 매 스텝 형식 오류 Observation이 되먹여지다가 `max_steps`에서 실패를 명시하며 종료한다. 상한이 없으면 이 대화는 비용을 소모하며 무한히 돈다.


In [ ]:
# (1) 정상 시나리오: 미등록 도구 호출 → 오류 관찰 → 답 확정
SCRIPTED_OUTPUTS = iter([
    'Thought: 검색 도구가 필요하다.\nAction: {"tool": "search", "input": "KTX 소요 시간"}',
    "Thought: 해당 도구가 없으므로 아는 범위에서 답한다.\nFinal Answer: 약 2시간 30분",
])
fake_llm = lambda messages: next(SCRIPTED_OUTPUTS)
print("반환값:", loop.react_loop("서울-부산 KTX 소요 시간은?", llm_fn=fake_llm, verbose=False))

# (2) 형식 위반 시나리오: Action도 Final Answer도 내지 않는 모델
stubborn_llm = lambda messages: "Thought: 아직 확신이 없다. 더 생각해야 한다."
print("반환값:", loop.react_loop("아무 질문", llm_fn=stubborn_llm, max_steps=3, verbose=False))


오프라인 테스트 전체를 실행한다. 전부 통과하면 W4 구현 요건은 충족된 것이다.


In [ ]:
!cd "{LABS}" && python -m pytest tests/test_week04.py -v


## 7. 프롬프트 개선 과제 — 형식 준수율 (핵심 과제)

시스템 프롬프트는 에이전트 거동의 사양서이며, 프롬프트를 측정하며 다듬는 것이 에이전트 개발의 실제 작업이다. 이 과제에서는 형식 준수율을 정의하고, 프롬프트를 개선해 수치를 올린다.

**형식 준수율** = 여러 질문에 대한 모델 출력 중 `parse_step` 판정이 `final` 또는 `action`인 비율. `neither` 판정은 계약 위반이다.

먼저 현재 계약(`loop.SYSTEM_PROMPT`)의 준수율을 측정한다. 변동을 드러내기 위해 temperature를 올려 측정한다.


In [ ]:
QUESTIONS = [
    "3 + 4는?",
    "오늘 서울 날씨는?",
    "1400의 29%는 얼마인가?",
    "트랜스포머 논문의 제목은?",
    "지금 환율로 100달러는 몇 원인가?",
    "파이썬 리스트와 튜플의 차이를 한 문장으로.",
]

def format_compliance(system_prompt, questions=QUESTIONS, temperature=0.8):
    """질문마다 1회 호출해 parse_step 판정이 final/action인 비율을 반환한다."""
    kinds = []
    for q in questions:
        out = llm.chat([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {q}"},
        ], temperature=temperature)
        kinds.append(loop.parse_step(out)[0])
    rate = sum(k in ("final", "action") for k in kinds) / len(kinds)
    print("판정 분포:", {k: kinds.count(k) for k in set(kinds)})
    return rate

baseline = format_compliance(loop.SYSTEM_PROMPT)
print(f"기준 준수율: {baseline:.0%}")


**과제:** 준수율을 기준치 이상으로 끌어올리는 `SYSTEM_PROMPT_V2`를 작성한다. 기준 프롬프트를 복사한 뒤 다음 개선 수단을 적용해 본다 (한 번에 하나씩 바꾸고 매번 측정하는 것이 원칙이다).

- **few-shot 예시** — 올바른 `Action` 턴과 올바른 `Final Answer` 턴의 예시를 각 1개 프롬프트에 포함한다 (W2 이론의 CoT 예시와 같은 원리다)
- **형식 재강조** — 지시를 출력 형식 명세 형태로 재서술한다 (허용되는 두 형식 외 금지임을 명시)
- **실패 사례 지시** — 3절에서 관찰한 위반(모델이 Observation을 씀)을 금지 사례로 명시한다

이미 기준 준수율이 100%이면 더 작은 모델(예: `.env`에서 `DOCQA_MODEL=ollama:llama3.2`)로 바꿔 기준을 다시 측정한다 — 작은 모델일수록 계약 위반이 잦아 프롬프트의 효과가 드러난다.


In [ ]:
SYSTEM_PROMPT_V2 = """\
# TODO: loop.SYSTEM_PROMPT를 바탕으로 개선안을 작성한다.
"""

improved = format_compliance(SYSTEM_PROMPT_V2)
print(f"개선 준수율: {improved:.0%}  (기준: {baseline:.0%})")


**기록:** 어떤 수단이 준수율을 올렸고 어떤 수단이 효과가 없었는가. 수단별 효과를 두세 문장으로 기록한다. 같은 수단이 모델에 따라 다르게 작동하는 것도 정상적인 관찰이다 — 프롬프트는 모델 종속적 사양서다.

개선안이 확정되면 `loop.py`의 `SYSTEM_PROMPT`를 교체하고 `pytest`로 회귀를 확인한다 (오프라인 테스트는 프롬프트와 무관하게 통과해야 정상이다 — 그 이유를 생각해 본다).


## 8. 연습문제

**8-1. 판정 우선순위와 포획 범위 (오프라인).** 한 출력에 `Final Answer:`와 `Action:`이 동시에 있으면 `parse_step`은 무엇을 반환하는가. 실행 전에 예측을 적고, 실행해 확인한다.


In [ ]:
mixed = 'Thought: 답을 알았다.\nFinal Answer: 42\nAction: {"tool": "search", "input": "x"}'
print(loop.parse_step(mixed))


`Final Answer`가 우선하지만, 정규식 `(.+)`가 `re.S` 플래그 때문에 뒤따르는 `Action:` 줄까지 포획하는 것을 확인할 수 있다. 반환된 답에 불필요한 꼬리가 붙는다 — 실전 파서의 전형적 취약점이다.

**과제:** `parse_step`을 수정해 Final Answer의 첫 줄만 답으로 취하도록 개선하고, 위 셀과 `pytest`를 다시 실행해 기존 테스트가 여전히 통과하는지 확인한다.

**8-2. 도구 부재 상황의 모델 거동 (모델 필요).** 성격이 다른 질문들로 `react_loop()`를 실행하고, 모델이 도구 호출을 시도하는 질문과 곧바로 답하는 질문의 차이를 관찰한다.


In [ ]:
for q in [
    "3 + 4는?",                    # 내부 지식으로 충분
    "오늘 서울 날씨는?",            # 외부 정보 필요
    "트랜스포머 논문의 제목은?",    # 경계 사례
]:
    print(f"\n{'='*50}\nQ: {q}")
    print("A:", loop.react_loop(q, verbose=False))


**질문:** 등록된 검색이 원시적 문자열 매칭(`text_search`)뿐인 현재 상태에서, 외부 지식이 필요한 질문에 모델이 내놓은 Final Answer는 신뢰할 수 있는가. 이 관찰이 W5(RAG)가 필요한 이유이며, 이론에서 다룬 접지(grounding)의 부재가 실행 수준에서 드러나는 지점이다.

**8-3. (선택) 루프 계측.** `react_loop()`를 수정해 최종 답과 함께 사용한 스텝 수를 반환하도록 바꾸고, 8-2의 세 질문이 각각 몇 스텝을 쓰는지 측정한다. 반환 형식 변경으로 `pytest`가 깨지면 테스트도 함께 고친다 — 인터페이스 변경이 호출부와 테스트에 파급되는 것을 직접 겪는 것이 목적이다.


## 9. 완료 기준

- [ ] `pytest tests/test_week04.py -v` 전체 통과
- [ ] `react_loop()`가 실제 모델로 Final Answer를 반환
- [ ] 3절 환각 실험을 재현하고 두 겹 방어(계약 문장, 파서·루프)의 역할을 설명할 수 있다
- [ ] 7절 `SYSTEM_PROMPT_V2` 작성 + 준수율 비교 + 수단별 효과 기록
- [ ] 연습문제 8-1의 파서 개선을 적용
- [ ] 미니 evalset 5문항 채점: `python demos/week04_react.py --eval` 결과 기록
- [ ] (선택) 터미널 데모: `python demos/week04_react.py "질문"`

오늘로 추론(W2)·도구(W3)·루프가 결합된 첫 완전한 에이전트가 완성되었다. 다음 주(W5) 실습은 `retriever.py`(임베딩 검색)를 추가해 원시 `text_search`를 교체한다.
